# Melatih LarisNet di Colab / Kaggle

Notebook ini melatih model prediksi penjualan **LarisNet** dengan GPU gratis, mengujinya terhadap
pembanding (rata-rata hari yang sama, LightGBM), lalu menghasilkan `model.onnx` + `meta.json` untuk aplikasi Laris.

1. **Colab:** Runtime → Change runtime type → **T4 GPU**. **Kaggle:** Settings → Accelerator → **GPU**, dan Internet **On**.
2. Jalankan sel dari atas ke bawah (sekitar 30–60 menit di T4).
3. Unduh `larisnet.zip` di akhir, lalu ganti isi folder `models/larisnet/` di repo dengan isinya.

In [ ]:
import os
REPO = "https://github.com/Epinnn14/laris.git"   # ganti kalau memakai salinan sendiri
if not os.path.exists("laris"):
    !git clone -q $REPO
%cd laris
!pip install -q -e "ml[train]"

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada (akan lambat)")

## Latih

Model lebih besar dan data lebih banyak daripada versi CPU di repo. Semua data adalah warung/kafe sintetis
(pola mingguan, gajian, Ramadan/Lebaran, libur nasional, hujan, promo, menu baru, stok habis) sehingga
tidak ada data pribadi yang dipakai.

In [ ]:
!python -m laris_ml.train --shops 3000 --val-shops 200 --steps 30000 --batch 512 \
    --d-model 128 --layers 4 --heads 8 --lr 1e-3 --out models/larisnet

## Uji pada toko yang belum pernah dilihat model

In [ ]:
!python -m laris_ml.evaluate --model models/larisnet --shops 150

In [ ]:
# Contoh: prediksi 4 minggu untuk satu menu warung contoh, dibanding penjualan sebenarnya
import numpy as np, matplotlib.pyplot as plt
from datetime import date
from laris_ml import synth, features as F
from laris_ml.forecast import Forecaster

fc = Forecaster("models/larisnet")
shop = synth.demo_shop(date(2026, 4, 20), business="warung_makan", location="kantor")  # mencakup Lebaran 2026
s = shop.series[0]
fr = s.frame()
start = fr.date.iloc[-F.HORIZON]
w = F.from_history(fr[fr.date < start], start, s.item.category, s.business,
                   future_closed=s.closed[-F.HORIZON:], future_promo=s.promo[-F.HORIZON:])
q = fc.predict([w])[0]
days = fr.date.iloc[-F.HORIZON:]
plt.figure(figsize=(11, 4))
plt.plot(fr.date.iloc[-90:-F.HORIZON], fr.qty.iloc[-90:-F.HORIZON], color="gray", label="riwayat")
plt.plot(days, fr.qty.iloc[-F.HORIZON:], "k.", label="sebenarnya")
plt.plot(days, q[:, 2], color="#b4441b", label="prediksi (tengah)")
plt.fill_between(days, q[:, 0], q[:, 4], color="#b4441b", alpha=0.2, label="kisaran 10–90%")
plt.title(s.item.name); plt.legend(); plt.show()

## Unduh model

In [ ]:
import shutil, os
shutil.make_archive("larisnet", "zip", "models/larisnet", ".")
try:
    from google.colab import files
    files.download("larisnet.zip")
except ImportError:
    os.makedirs("/kaggle/working", exist_ok=True)
    shutil.copy("larisnet.zip", "/kaggle/working/larisnet.zip")
    print("Kaggle: ambil larisnet.zip dari tab Output")

## Pasang di aplikasi

Ekstrak `model.onnx`, `meta.json`, `eval.json`, `eval.md` ke `models/larisnet/` di repo, lalu commit.
Aplikasi (dan image Docker) otomatis memakai model baru. `model.pt` tidak perlu di-commit.